In [ ]:
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding
)

# Load your dataset

In [ ]:
from datasets import load_dataset

# Replace 'stanfordnlp/IMDb' with the correct organization/namespace if different
dataset = load_dataset(
    "stanfordnlp/IMDb", revision="e6281661ce1c48d982bc483cf8a173c1bbeb5d31"
)

train_data = dataset["train"]

eval_data = dataset.get(
    "test",
    dataset.get("validation")
)

print(train_data[0])

# Automatically detect the number of classes

In [ ]:
num_labels=train_data.features["label"].num_classes
print("number of classes",num_labels)

# Load the tokenizer

In [ ]:
#Automatically loads the appropriate tokenizer class.Loads the tokenizer associated with a pretrained model
tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Tokenize the dataset 
We take sentences from our dataset, convert them into numerical token IDs using the tokenizer, and replace the original text with the tokenized data.

In [ ]:
def tokenize_function(batch):#batch contain 16 sentences each batch
    return tokenizer(
        batch["text"],
        truncation=True,   #Cuts sentences that are too long.
        max_length=256     #Sets the maximum number of tokens.
    )#If a sentence produces 300 tokens, the tokenizer cuts it to 256 tokens.

#.map() applies your function to the dataset and creates a new dataset containing the results.
train_data=train_data.map(
    tokenize_function,
    batched=True,#with batch Hugging Face sends multiple sentences together like batch_size=16 16sentences.
    remove_columns=["text"]#text column sentences is removed bc we already have ids for the texts sentences of each
)
val_data=val_data.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

# Create DataLoaders
Remember: DataLoader collects the examples; the collator prepares them into a batch.

In [ ]:
#DataCollatorWithPadding makes sentences in the same batch equal in length by adding padding.
#It automatically adds padding tokens to shorter sequences so every sequence in the batch has the same length.
collator=DataCollatorWithPadding(tokenizer=tokenizer)#give the collator our previously loaded tokenizer.

train_loader=DataLoader(
    train_data,#1000 examples produce 63 batches when batch size is 16. The last batch contains 8 examples.
    batch_size=16,
    shuffle=True,
    collat_fn=collator#This connects DataLoader with DataCollatorWithPadding.
)
eval_loader=DataLoder(
    eval_data,
    batch_size=16
    collat_fn=collator
)


# Load the pretrained model
- AutoModel: Automatically loads the correct pretrained model architecture.
- ForSequenceClassification: Prepares that model for sentence classification.
- from_pretrained(): Loads previously learned model weights.

In [ ]:
model=AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased",num_labels=2)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Optimizer

In [ ]:
optimizer=torch.optim.AdamW(model.parameters(),lr=2e-5)

# Fine-tune the model training loop 

In [ ]:
for epoch in range(25):
    model.train()
    total_loss=0

    for batch in train_loader:
        bathc={key:value.to(device)
               for key,value in batch.items()}
        
        optimizer.zero_grad()
        output=model(**batch)
        loss=output.loss
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    print(
        f"Epoch {epoch + 1}, "
        f"Loss: {total_loss / len(train_loader):.4f}"
    )

# Evaluate the model

In [ ]:
model.eval()
correct=0
total=0

with torch.no_grad():#"Don't calculate gradients. I only want predictions."
    for batch in eval_loader:
        batch={
            key:value.to(device)
            for key,value in batch.items()}
        outputs=model(**batch)
        predictions=output.logits.argmax(dim=1)
        correct+=(predictions==batch["labels"]).sum().item()
        total+=batch["labels"].size(0)

accuracy = correct / total

print(f"Accuracy: {accuracy:.4f}")

# Save your fine-tuned model

In [ ]:
model.save_pretrained("my_finetuned_model")

tokenizer.save_pretrained("my_finetuned_model")

# Make a prediction

In [ ]:
model.eval()

text = "This movie was absolutely fantastic!"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_LENGTH
).to(device)

with torch.no_grad():

    outputs = model(**inputs)

prediction = outputs.logits.argmax(dim=1).item()

print("Predicted class:", prediction)